# Density Sweep trên Subsampled ML-20M — M1 / M7 / R1

Notebook này **tái dùng nguyên code & data gốc** của paper *LLM-MovieLens* (repo `llm-movielens`)
và chỉ thêm một lớp điều phối: tạo subsample theo mật độ → chạy 3 phương pháp **M1 / M7 / R1** →
lưu checkpoint + log đầy đủ → tổng hợp + vẽ đường **∆R1(d)**.

**Khớp với code gốc để kiểm chứng được:**
- **M1 / M7** chạy bằng `code/benchmark/run_experiment.py` (LightGCN / LightGCN-SF), *đúng CLI* của
  `scripts/run_subsampled_density_sweep.sh`.
- **R1 (RLMRec-gene)** train bằng `external/RLMRec/encoder/train_encoder.py`, đúng như sweep gốc.
- **Đánh giá R1 dùng cùng `evaluate.compute_metrics`** (full-ranking, mask train-items, sum `layer_num`
  lớp) như M1/M7 — *cùng evaluator* nên NDCG@10/Recall@10/MRR so sánh trực tiếp được
  (mô phỏng `scripts/eval_ml20m_sub163_r1.py`).

Thiết kế: **4 mật độ × 3 method × 3 seed = 36 ô**. Idempotent (resume được qua `run_records.jsonl`).

> Chạy tuần tự từ trên xuống. Đặt notebook ở **thư mục gốc repo** (`llm-movielens/`); nếu để nơi khác,
> sửa biến `REPO` ở ô **0 · Cấu hình**.

# -1. Tải data

In [ ]:
import shutil
from pathlib import Path
from huggingface_hub import snapshot_download

In [ ]:
HF_REPO = "anonyauthor4review/llm-movielens"

DOWNLOAD_DIR = Path("./llm-movielens")

snapshot_download(
    repo_id=HF_REPO,
    repo_type="dataset",
    local_dir=str(DOWNLOAD_DIR),
)

print("✓ Download complete")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 84 files:   0%|          | 0/84 [00:00<?, ?it/s]

✓ Download complete


In [ ]:
PROCESSED_DIR = Path(
    "./code/benchmark/data/processed"
)

EMBED_DIR = Path(
    "./code/embedding_generator/output/bge-large-en-v1.5"
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EMBED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
src_benchmark = DOWNLOAD_DIR / "benchmark_splits" / "ml20m"

for file in src_benchmark.iterdir():
    if file.is_file():
        shutil.copy2(file, PROCESSED_DIR / file.name)

print("✓ benchmark_splits/ml20m copied")

✓ benchmark_splits/ml20m copied


In [ ]:
src_embed = (
    DOWNLOAD_DIR
    / "embeddings"
    / "ml20m"
    / "bge-large-en-v1.5"
)

for file in src_embed.iterdir():
    if file.is_file():
        shutil.copy2(file, EMBED_DIR / file.name)

print("✓ embeddings copied")

✓ embeddings copied


In [ ]:
PROFILE_DIR = Path(
    "./data/movie_profiles"
)

PROFILE_DIR.mkdir(parents=True, exist_ok=True)

profile_json = (
    DOWNLOAD_DIR
    / "profiles"
    / "ml20m"
    / "claude-haiku-4-5"
    / "movie_profiles.json"
)

if profile_json.exists():
    shutil.copy2(
        profile_json,
        PROFILE_DIR / "movie_profiles.json"
    )
    print("✓ movie_profiles.json copied")

✓ movie_profiles.json copied


In [ ]:
print("\n===== VERIFY =====")

for p in [
    PROCESSED_DIR,
    EMBED_DIR,
]:
    print(f"\n{p}")
    for f in sorted(p.iterdir()):
        print("  ", f.name)

print("\nSetup finished.")


===== VERIFY =====

code/benchmark/data/processed
   item_map.json
   stats.json
   test.csv
   train.csv
   user_map.json
   val.csv

code/embedding_generator/output/bge-large-en-v1.5
   bert_title_embeddings.npy
   combined_features.npy
   combined_full.npy
   embedding_metadata.json
   genome_embeddings.npy
   mood_vectors.npy
   movie_id_index.json
   profile_embeddings.npy
   theme_matrix.npy
   theme_vocabulary.json

Setup finished.


In [ ]:
from pathlib import Path

print(Path("code/benchmark/data/processed").exists())
print(Path("code/embedding_generator/output/bge-large-en-v1.5").exists())

True
True


In [ ]:
from pathlib import Path
import numpy as np
import json

EMB = Path("code/embedding_generator/output/bge-large-en-v1.5")

print("profile:", np.load(EMB/"profile_embeddings.npy").shape)
print("mood:", np.load(EMB/"mood_vectors.npy").shape)

with open(EMB/"movie_id_index.json","r") as f:
    idx = json.load(f)

print("movie_id_index:", len(idx))

profile: (10381, 1024)
mood: (10381, 10)
movie_id_index: 10381


## 0 · Cấu hình

Các biến hay đổi: `DENSITIES`, `SEEDS`, `METHODS`, `DEVICE`, `RERUN`, và (nếu muốn so khớp dung lượng
với M7) `R1_EMBEDDING_SIZE = 128`.

In [ ]:
import os, sys, json, time, shutil, subprocess, platform
from pathlib import Path

# ── Đường dẫn gốc ──────────────────────────────────────────────────────────────
# Tự dò REPO: ưu tiên cwd nếu có code/benchmark, nếu không thì đi lên tìm.
def _autodetect_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "code" / "benchmark" / "run_experiment.py").exists():
            return cand
    return here

REPO = _autodetect_repo()                       # ← sửa tay nếu cần: REPO = Path("/path/to/llm-movielens")
BENCH = REPO / "code" / "benchmark"             # 'code' là symlink → 'src' trong repo
RLMREC_ROOT = BENCH / "external" / "RLMRec"

# Data + embedding nguồn (ML-20M đã tiền xử lý + embedding bge)
FULL_PROCESSED_DIR = BENCH / "data" / "processed"
EMBEDDING_DIR      = BENCH.parent / "embedding_generator" / "output" / "bge-large-en-v1.5"
ENCODER_NAME       = EMBEDDING_DIR.name          # "bge-large-en-v1.5" → dùng trong đường dẫn output

# ── Thí nghiệm ─────────────────────────────────────────────────────────────────
DENSITIES = [600, 300, 150, 75]                  # target int/item (10-core sẽ làm cao hơn — báo cáo achieved)
SEEDS     = [42, 123, 456]                       # 3 eval seeds (tách biệt subsample-seed=42)
METHODS   = ["M1", "M7", "R1"]                   # M1=LightGCN, M7=+profile+mood, R1=RLMRec-gene
SUBSAMPLE_SEED = 42                              # protocol seed (giữ splits tái lập 100%)
RERUN = False                                    # True = bỏ qua run_records, chạy lại tất cả

# Device: "auto" | "cuda" | "mps" | "cpu"
DEVICE = "auto"

# ── Bootstrap source code (Colab / máy trống) ───────────────────────────────────
# Nếu thiếu source repo, ô "0.5 · Bootstrap" sẽ tự nạp theo thứ tự ưu tiên:
#   1) LOCAL_REPO_ZIP (nếu bạn đã upload zip repo lên máy)  2) git clone REPO_GIT_URL
REPO_GIT_URL  = "https://github.com/anonyauthor4review-png/llm-movielens.git"
REPO_BRANCH   = None                             # None = nhánh mặc định
LOCAL_REPO_ZIP = None                            # vd "/content/llm-movielens-main.zip" (None = tự dò)
AUTO_PIP_INSTALL = True                          # cài tqdm/pyyaml/matplotlib... nếu thiếu
FORCE_BOOTSTRAP = False                          # True = nạp lại source dù đã có

# R1 (RLMRec-gene) hparams — protocol sub163 cùng-domain (mặc định upstream)
R1_LAYER_NUM      = 3
R1_MASK_RATIO     = 0.1
R1_RECON_WEIGHT   = 0.1
R1_RE_TEMPERATURE = 0.2
R1_REG_WEIGHT     = 1.0e-7
R1_EMBEDDING_SIZE = 32                            # đặt 128 nếu muốn so khớp dung lượng với M7

# ── Nơi chứa toàn bộ output của sweep ──────────────────────────────────────────
EXP_ROOT = REPO / "experiments" / "density_sweep_sub20m"
DATA_OUT_ROOT  = EXP_ROOT / "data"
CKPT_ROOT      = EXP_ROOT / "checkpoints"
RESULTS_ROOT   = EXP_ROOT / "results"
LOG_ROOT       = EXP_ROOT / "logs"
R1_MODELCONF   = EXP_ROOT / "r1_modelconf"
RUN_RECORDS    = EXP_ROOT / "run_records.jsonl"
for d in (DATA_OUT_ROOT, CKPT_ROOT, RESULTS_ROOT, LOG_ROOT, R1_MODELCONF):
    d.mkdir(parents=True, exist_ok=True)

def resolve_device(dev: str) -> str:
    if dev != "auto":
        return dev
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"

DEVICE = resolve_device(DEVICE)

print(f"REPO              = {REPO}")
print(f"BENCH             = {BENCH}")
print(f"RLMREC_ROOT       = {RLMREC_ROOT}")
print(f"FULL_PROCESSED    = {FULL_PROCESSED_DIR}")
print(f"EMBEDDING_DIR     = {EMBEDDING_DIR}  (encoder={ENCODER_NAME})")
print(f"EXP_ROOT          = {EXP_ROOT}")
print(f"DEVICE            = {DEVICE}")
print(f"DENSITIES/SEEDS/METHODS = {DENSITIES} / {SEEDS} / {METHODS}")
print(f"R1 embedding_size = {R1_EMBEDDING_SIZE}")


REPO              = /content
BENCH             = /content/code/benchmark
RLMREC_ROOT       = /content/code/benchmark/external/RLMRec
FULL_PROCESSED    = /content/code/benchmark/data/processed
EMBEDDING_DIR     = /content/code/embedding_generator/output/bge-large-en-v1.5  (encoder=bge-large-en-v1.5)
EXP_ROOT          = /content/experiments/density_sweep_sub20m
DEVICE            = cpu
DENSITIES/SEEDS/METHODS = [600, 300, 150, 75] / [42, 123, 456] / ['M1', 'M7', 'R1']
R1 embedding_size = 32


## 0.5 · Bootstrap — tự động nạp source code repo

Trên Colab/máy trống thường **chỉ có data + embedding** mà thiếu source code (`run_experiment.py`,
`RLMRec`, …). Ô này tự nạp source vào đúng `code/benchmark/...` **mà không đụng** `data/processed/`
và `embedding_generator/output/` bạn đã có. Thứ tự ưu tiên: (1) zip repo bạn đã upload, (2) `git clone`.

In [ ]:
def _need_source() -> bool:
    must = [BENCH / "run_experiment.py",
            RLMREC_ROOT / "encoder" / "train_encoder.py",
            RLMREC_ROOT / "encoder" / "data_utils" / "data_handler_general_cf.py",
            RLMREC_ROOT / "encoder" / "config" / "modelconf" / "lightgcn_gene.yml"]
    return not all(p.exists() for p in must)

def _find_clone_root(base: Path) -> Path:
    """Tìm thư mục gốc repo (chứa src/benchmark hoặc code/benchmark) trong 'base'."""
    cands = [base, *[p for p in base.iterdir() if p.is_dir()]] if base.is_dir() else [base]
    for c in cands:
        if (c / "src" / "benchmark" / "run_experiment.py").exists():
            return c
        if (c / "code" / "benchmark" / "run_experiment.py").exists():
            return c
    raise FileNotFoundError(f"Không thấy src/benchmark hoặc code/benchmark trong {base}")

def _merge_source(clone_root: Path):
    """Copy source vào /content/code/... (merge, KHÔNG xoá; giữ data + embedding sẵn có)."""
    src_bench = clone_root / "src" / "benchmark"
    if not src_bench.exists():
        src_bench = (clone_root / "code" / "benchmark").resolve()   # 'code' là symlink → src
    src_root = src_bench.parent                                     # .../src (hoặc resolved code)
    BENCH.mkdir(parents=True, exist_ok=True)
    # 1) toàn bộ module benchmark (gồm external/RLMRec) → code/benchmark
    shutil.copytree(src_bench, BENCH, dirs_exist_ok=True)
    # 2) code của embedding_generator (KHÔNG có .npy của bạn nên an toàn) → code/embedding_generator
    eg = src_root / "embedding_generator"
    if eg.exists():
        shutil.copytree(eg, BENCH.parent / "embedding_generator", dirs_exist_ok=True)
    print(f"  ✓ đã merge source từ {src_bench}")

def _pip(pkgs):
    import importlib.util
    missing = [p for p in pkgs if importlib.util.find_spec(p.replace("-", "_")) is None]
    # tên import khác tên pip cho vài gói:
    alias = {"yaml": "pyyaml", "sklearn": "scikit-learn"}
    need = []
    for imp in ["yaml", "tqdm", "matplotlib", "scipy", "sklearn", "pandas", "numpy"]:
        import importlib.util as _u
        if _u.find_spec(imp) is None:
            need.append(alias.get(imp, imp))
    if need:
        print(f"  pip install {' '.join(need)} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=False)
    try:
        import torch  # noqa
    except Exception:
        print("  ⚠ torch chưa có — Colab thường có sẵn; nếu không: pip install torch")

if FORCE_BOOTSTRAP or _need_source():
    print("Thiếu source — đang bootstrap...")
    got = False

    # (1) zip repo đã upload
    zip_candidates = []
    if LOCAL_REPO_ZIP:
        zip_candidates.append(Path(LOCAL_REPO_ZIP))
    zip_candidates += sorted(Path("/content").glob("llm-movielens*.zip")) if Path("/content").exists() else []
    zip_candidates += sorted(Path.cwd().glob("llm-movielens*.zip"))
    for zp in zip_candidates:
        if zp.exists():
            print(f"  giải nén {zp} ...")
            tmp = Path("/tmp/_llmml_zip"); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
            shutil.unpack_archive(str(zp), str(tmp))
            try:
                _merge_source(_find_clone_root(tmp)); got = True; break
            except FileNotFoundError as e:
                print(f"    (bỏ qua zip: {e})")

    # (2) git clone
    if not got:
        tmp = Path("/tmp/_llmml_clone"); shutil.rmtree(tmp, ignore_errors=True)
        cmd = ["git", "clone", "--depth", "1"]
        if REPO_BRANCH: cmd += ["--branch", REPO_BRANCH]
        cmd += [REPO_GIT_URL, str(tmp)]
        print(f"  {' '.join(cmd)}")
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[-800:])
            raise RuntimeError("git clone thất bại. Hãy upload zip repo lên /content rồi chạy lại, "
                               "hoặc đặt LOCAL_REPO_ZIP / REPO_GIT_URL ở ô Cấu hình.")
        _merge_source(_find_clone_root(tmp)); got = True

    if AUTO_PIP_INSTALL:
        _pip([])
    print("Bootstrap xong." if not _need_source() else "⚠ Vẫn thiếu vài file — kiểm tra lại nguồn.")
else:
    print("Source đã có — bỏ qua bootstrap.")
    if AUTO_PIP_INSTALL:
        _pip([])


Source đã có — bỏ qua bootstrap.


## 1 · Logging

Một logger tổng (`master_*.log`) cho toàn sweep; mỗi ô có log riêng dưới `logs/sub{d}/`.

In [ ]:
import logging
from datetime import datetime

MASTER_LOG = LOG_ROOT / f"master_{datetime.now():%Y%m%d-%H%M%S}.log"
logger = logging.getLogger("density_sweep")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%H:%M:%S")
_sh = logging.StreamHandler(sys.stdout); _sh.setFormatter(_fmt); logger.addHandler(_sh)
_fh = logging.FileHandler(MASTER_LOG);   _fh.setFormatter(_fmt); logger.addHandler(_fh)
logger.propagate = False
logger.info(f"Master log → {MASTER_LOG}")


17:14:54 [INFO] Master log → /content/experiments/density_sweep_sub20m/logs/master_20260614-171454.log


## 2 · Kiểm tra môi trường

Xác nhận: (a) scripts gốc tồn tại, (b) data ML-20M đã tiền xử lý, (c) embedding bge sẵn sàng và
**`movie_id_index.json` nằm cùng thư mục** với `profile_embeddings.npy` (`FeatureLoader` đọc cả 3 từ
cùng `EMBEDDING_DIR`). Nếu artifact HF tách file, ô này tự **symlink** chúng về cùng chỗ.

In [ ]:
def _ok(b): return "✓" if b else "✗"

problems = []

# (a) scripts gốc
RUN_EXP     = BENCH / "run_experiment.py"
TRAIN_ENC   = RLMREC_ROOT / "encoder" / "train_encoder.py"
DATA_HANDLER = RLMREC_ROOT / "encoder" / "data_utils" / "data_handler_general_cf.py"
LIGHTGCN_GENE_YML = RLMREC_ROOT / "encoder" / "config" / "modelconf" / "lightgcn_gene.yml"
for p in (RUN_EXP, TRAIN_ENC, DATA_HANDLER, LIGHTGCN_GENE_YML):
    exists = p.exists()
    print(f"  {_ok(exists)} {p.relative_to(REPO) if exists else p}")
    if not exists: problems.append(f"thiếu {p}")

# (b) data ML-20M đã tiền xử lý
print("\n[data ML-20M processed]")
need_data = ["train.csv", "val.csv", "test.csv", "item_map.json", "stats.json"]
for f in need_data:
    p = FULL_PROCESSED_DIR / f
    print(f"  {_ok(p.exists())} {p}")
    if not p.exists(): problems.append(f"thiếu {p}")

# (c) embedding bge — đảm bảo movie_id_index.json + *.npy cùng EMBEDDING_DIR
print("\n[embedding bge]")
RESOLVED_EMBED_DIR = EMBEDDING_DIR
need_emb = {"profile_embeddings.npy": None, "mood_vectors.npy": None, "movie_id_index.json": None}
for f in list(need_emb):
    p = RESOLVED_EMBED_DIR / f
    if not p.exists():
        # thử symlink từ thư mục cha output/ (một số artifact HF tách file)
        parent_candidate = RESOLVED_EMBED_DIR.parent / f
        if parent_candidate.exists():
            try:
                p.symlink_to(parent_candidate)
                print(f"  ↪ symlink {f} ← {parent_candidate}")
            except Exception as e:
                print(f"  (symlink {f} thất bại: {e}; hãy copy thủ công)")
    ok = p.exists()
    if ok:
        try:
            import numpy as _np
            shape = _np.load(p).shape if f.endswith(".npy") else f"(json, {len(json.loads(p.read_text()))} ids)"
        except Exception:
            shape = "?"
    else:
        shape = "—"
    print(f"  {_ok(ok)} {p}   shape={shape}")
    if not ok: problems.append(f"thiếu {p}")

print()
if problems:
    logger.warning("THIẾU %d mục — hãy bổ sung trước khi chạy sweep:", len(problems))
    for x in problems: logger.warning("   - %s", x)
else:
    logger.info("Môi trường OK: data + embedding + scripts sẵn sàng.")


  ✓ code/benchmark/run_experiment.py
  ✓ code/benchmark/external/RLMRec/encoder/train_encoder.py
  ✓ code/benchmark/external/RLMRec/encoder/data_utils/data_handler_general_cf.py
  ✓ code/benchmark/external/RLMRec/encoder/config/modelconf/lightgcn_gene.yml

[data ML-20M processed]
  ✓ /content/code/benchmark/data/processed/train.csv
  ✓ /content/code/benchmark/data/processed/val.csv
  ✓ /content/code/benchmark/data/processed/test.csv
  ✓ /content/code/benchmark/data/processed/item_map.json
  ✓ /content/code/benchmark/data/processed/stats.json

[embedding bge]
  ✓ /content/code/embedding_generator/output/bge-large-en-v1.5/profile_embeddings.npy   shape=(10381, 1024)
  ✓ /content/code/embedding_generator/output/bge-large-en-v1.5/mood_vectors.npy   shape=(10381, 10)
  ✓ /content/code/embedding_generator/output/bge-large-en-v1.5/movie_id_index.json   shape=(json, 10381 ids)

17:14:56 [INFO] Môi trường OK: data + embedding + scripts sẵn sàng.


## 3 · Subsample

Sinh `data/processed_ml20m_sub{d}/` cho mỗi mật độ. Mỗi bản gồm `train/val/test.csv` (ID liên tục),
`stats.json`, `user_map.json`, `subsampled_to_ml20m_movieId.json`, và `item_map.json`.

**Điểm "faithful với paper" (note #1 trong README):** `FeatureLoader` của repo căn embedding qua
`{movieId → genome_row}`. Script subsample gốc ghi `item_map.json = {processed_id: sub_id}` (sai khoá
cho loader). Ở đây ta ghi **`item_map.json = {ml20m_movieId: sub_id}`** để loader chuẩn căn *đúng*
embedding ML-20M mà **không sửa lõi**. Cột `emb_matched` ở bảng dưới phải ≈ `n_items`.

In [ ]:
import numpy as np
import pandas as pd

def _kcore(df: pd.DataFrame, k: int = 10) -> pd.DataFrame:
    while True:
        u_count = df.groupby("userId").size()
        i_count = df.groupby("movieId").size()
        u_keep = u_count[u_count >= k].index
        i_keep = i_count[i_count >= k].index
        new = df[df.userId.isin(u_keep) & df.movieId.isin(i_keep)]
        if len(new) == len(df):
            return new
        df = new

def subsample_density(target_int_per_item: int, out_dir: Path, subsample_seed: int = SUBSAMPLE_SEED):
    """User-uniform subsample (giữ long-tail) + 10-core. item_map.json dùng KHOÁ = ml20m_movieId."""
    out_dir.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(FULL_PROCESSED_DIR / "train.csv")
    val   = pd.read_csv(FULL_PROCESSED_DIR / "val.csv")
    test  = pd.read_csv(FULL_PROCESSED_DIR / "test.csv")

    # base item_map: {ml20m_movieId(str): processed_id} → đảo: processed_id → ml20m_movieId
    base_item_map = json.loads((FULL_PROCESSED_DIR / "item_map.json").read_text())
    proc_to_ml20m = {int(v): int(k) for k, v in base_item_map.items()}

    # (2) user-uniform sample tới target tổng tương tác
    target_total = target_int_per_item * train.movieId.nunique()
    rng = np.random.default_rng(subsample_seed)
    u_int = train.groupby("userId").size()
    users = u_int.index.values.copy(); rng.shuffle(users)
    cum = u_int.loc[users].cumsum().values
    n_keep = int(np.searchsorted(cum, target_total) + 1)
    n_keep = min(n_keep, len(users))
    kept = set(users[:n_keep].tolist())

    # (3) lọc 3 split + 10-core trên train
    train = train[train.userId.isin(kept)].reset_index(drop=True)
    val   = val[val.userId.isin(kept)].reset_index(drop=True)
    test  = test[test.userId.isin(kept)].reset_index(drop=True)
    train = _kcore(train, k=10).reset_index(drop=True)
    items_kept = set(train.movieId.unique().tolist())
    users_kept = set(train.userId.unique().tolist())
    val  = val[val.userId.isin(users_kept) & val.movieId.isin(items_kept)].reset_index(drop=True)
    test = test[test.userId.isin(users_kept) & test.movieId.isin(items_kept)].reset_index(drop=True)

    # (4) remap về ID liên tục (processed_id → sub_id)
    item_old_to_new = {int(o): i for i, o in enumerate(sorted(items_kept))}
    user_old_to_new = {int(o): i for i, o in enumerate(sorted(users_kept))}
    for df in (train, val, test):
        df["userId"]  = df["userId"].map(user_old_to_new).astype("int64")
        df["movieId"] = df["movieId"].map(item_old_to_new).astype("int64")

    # (5) bridge cho RLMRec/feature loader: sub_id → ml20m_movieId
    sub_to_ml20m = {str(new): proc_to_ml20m[old] for old, new in item_old_to_new.items() if old in proc_to_ml20m}

    # (6) GHI item_map.json = {ml20m_movieId(str): sub_id}  ← khoá đúng cho FeatureLoader
    item_map_for_loader = {str(proc_to_ml20m[old]): new for old, new in item_old_to_new.items() if old in proc_to_ml20m}

    train.to_csv(out_dir / "train.csv", index=False)
    val.to_csv(out_dir / "val.csv", index=False)
    test.to_csv(out_dir / "test.csv", index=False)
    (out_dir / "item_map.json").write_text(json.dumps(item_map_for_loader))
    (out_dir / "user_map.json").write_text(json.dumps({str(k): v for k, v in user_old_to_new.items()}))
    (out_dir / "subsampled_to_ml20m_movieId.json").write_text(json.dumps(sub_to_ml20m))

    n_u = int(train.userId.max() + 1); n_i = int(train.movieId.max() + 1)
    counts = train.groupby("movieId").size()
    achieved = float(len(train) / n_i)
    stats = {
        "n_users": n_u, "n_items": n_i,
        "n_train": int(len(train)), "n_val": int(len(val)), "n_test": int(len(test)),
        "density": float(len(train) / (n_u * n_i)),
        "n_cold_items": int((counts < 10).sum()),
        "n_medium_items": int(((counts >= 10) & (counts < 50)).sum()),
        "n_warm_items": int((counts >= 50).sum()),
        "subsample_strategy": "user-uniform random + 10-core",
        "subsample_seed": subsample_seed,
        "target_int_per_item": target_int_per_item,
        "achieved_int_per_item": achieved,
        "items_per_user": float(len(train) / n_u),
        "emb_matched": len(sub_to_ml20m),
    }
    (out_dir / "stats.json").write_text(json.dumps(stats, indent=2))
    return stats

# Chạy subsample cho từng mật độ (idempotent: nếu đã có stats.json + đủ file → bỏ qua)
DENSITY_DIRS = {}
rows = []
for d in DENSITIES:
    out_dir = DATA_OUT_ROOT / f"processed_ml20m_sub{d}"
    DENSITY_DIRS[d] = out_dir
    need = ["train.csv", "val.csv", "test.csv", "item_map.json", "stats.json",
            "subsampled_to_ml20m_movieId.json"]
    if (not RERUN) and all((out_dir / f).exists() for f in need):
        stats = json.loads((out_dir / "stats.json").read_text())
        logger.info(f"[sub{d}] đã có — bỏ qua (achieved={stats['achieved_int_per_item']:.1f} int/item)")
    else:
        logger.info(f"[sub{d}] subsample target={d} int/item ...")
        stats = subsample_density(d, out_dir)
        logger.info(f"[sub{d}] done: users={stats['n_users']:,} items={stats['n_items']:,} "
                    f"train={stats['n_train']:,} achieved={stats['achieved_int_per_item']:.1f} "
                    f"emb_matched={stats['emb_matched']}/{stats['n_items']}")
    rows.append({"target": d, "n_users": stats["n_users"], "n_items": stats["n_items"],
                 "n_train": stats["n_train"], "achieved_int_per_item": round(stats["achieved_int_per_item"], 1),
                 "items_per_user": round(stats["items_per_user"], 1),
                 "emb_matched": stats["emb_matched"]})

import pandas as pd
density_table = pd.DataFrame(rows).set_index("target")
print()
print(density_table.to_string())


17:15:10 [INFO] [sub600] subsample target=600 int/item ...
17:15:27 [INFO] [sub600] done: users=65,898 items=9,397 train=5,940,763 achieved=632.2 emb_matched=9397/9397
17:15:27 [INFO] [sub300] subsample target=300 int/item ...
17:15:38 [INFO] [sub300] done: users=33,027 items=8,370 train=2,963,513 achieved=354.1 emb_matched=8370/8370
17:15:38 [INFO] [sub150] subsample target=150 int/item ...
17:15:44 [INFO] [sub150] done: users=16,266 items=6,967 train=1,472,753 achieved=211.4 emb_matched=6967/6967
17:15:44 [INFO] [sub75] subsample target=75 int/item ...
17:15:50 [INFO] [sub75] done: users=8,189 items=5,293 train=726,273 achieved=137.2 emb_matched=5293/5293

        n_users  n_items  n_train  achieved_int_per_item  items_per_user  emb_matched
target                                                                               
600       65898     9397  5940763                  632.2            90.2         9397
300       33027     8370  2963513                  354.1            89.7   

## 4 · Runner M (M1 / M7)

Gọi `code/benchmark/run_experiment.py` đúng CLI của `scripts/run_subsampled_density_sweep.sh`:
- **M1** → `--model lightgcn   --features none`
- **M7** → `--model lightgcn_sf --features llm_prof_mood`

Output ghi vào `results/sub{d}/{cfg}/{enc}/seed-{seed}/results.json` và checkpoint
`checkpoints/sub{d}/{cfg}/{enc}/seed-{seed}/best_model.pt` (+`training_state.pt`) — đúng layout repo.

In [ ]:
M_SPEC = {
    "M1": ("lightgcn",    "none"),
    "M7": ("lightgcn_sf", "llm_prof_mood"),
}
CFG_LOWER = {"M1": "m1", "M7": "m7", "R1": "r1"}

def run_M_cell(method: str, d: int, seed: int) -> dict:
    """Chạy 1 ô M1/M7 qua run_experiment.py. Trả về dict gồm metrics + đường dẫn."""
    model, features = M_SPEC[method]
    data_dir   = DENSITY_DIRS[d]
    results_dir = RESULTS_ROOT / f"sub{d}"
    ckpt_dir    = CKPT_ROOT / f"sub{d}"
    log_dir     = LOG_ROOT / f"sub{d}"; log_dir.mkdir(parents=True, exist_ok=True)
    log_path    = log_dir / f"{method}-seed{seed}.log"

    cmd = [sys.executable, "run_experiment.py",
           "--model", model, "--features", features, "--seed", str(seed),
           "--data-dir", str(data_dir),
           "--embedding-dir", str(EMBEDDING_DIR),
           "--results-dir", str(results_dir),
           "--checkpoint-dir", str(ckpt_dir),
           "--device", DEVICE]
    t0 = time.time()
    with open(log_path, "w") as lf:
        proc = subprocess.run(cmd, cwd=str(BENCH), stdout=lf, stderr=subprocess.STDOUT, text=True)
    wall = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"{method} sub{d} seed{seed} FAILED (exit={proc.returncode}); xem {log_path}")

    exp_name = f"{CFG_LOWER[method]}/{ENCODER_NAME}/seed-{seed}"
    res_json = results_dir / exp_name / "results.json"
    ckpt     = ckpt_dir / exp_name / "best_model.pt"
    res = json.loads(res_json.read_text())
    tm = res.get("test_metrics", {})
    # train_time tổng: cộng history trong training_state.pt
    train_time = wall
    ts = ckpt_dir / exp_name / "training_state.pt"
    try:
        import torch
        st = torch.load(ts, map_location="cpu", weights_only=False)
        train_time = float(sum(e.get("train_time", 0.0) for e in st.get("history", [])))
    except Exception:
        pass
    return {
        "ndcg10": tm.get("NDCG@10"), "recall10": tm.get("Recall@10"), "mrr": tm.get("MRR"),
        "ndcg20": tm.get("NDCG@20"), "recall20": tm.get("Recall@20"),
        "best_epoch": res.get("best_epoch"),
        "train_time_sec": round(train_time, 1), "wall_sec": round(wall, 1),
        "results_path": str(res_json), "ckpt_path": str(ckpt),
    }


## 5 · Runner R1 (RLMRec-gene)

Ba bước, mô phỏng `scripts/prepare_rlmrec_ml20m_sub163.py` + `train_encoder.py` + `eval_ml20m_sub163_r1.py`:

1. **`prepare_r1_data(d)`** — dựng `external/RLMRec/data/ml20m_sub{d}_ours/`: `trn/val/tst_mat.pkl`,
   `itm_emb_np.pkl` (căn từ embedding ML-20M qua `subsampled_to_ml20m_movieId.json`),
   `usr_emb_np.pkl` (mean-pool, unit-norm), `itm_prf.pkl`, `usr_prf.pkl`.
2. **Patch** (idempotent): thêm nhánh `predir` trong `data_handler_general_cf.py`; sinh modelconf YAML
   riêng cho mỗi (d, seed) (đặt `ckpt_out_dir`/`resume` để có `best_model.pt`+`training_state.pt`).
   *Không sửa* YAML upstream — ta dùng `--model-conf`.
3. **Train** bằng `encoder/train_encoder.py` (native `.pth` lưu ở `encoder/checkpoint/lightgcn_gene/`),
   rồi **đánh giá bằng `evaluate.compute_metrics`** (cùng evaluator M1/M7) để có NDCG@10/Recall@10/MRR
   so sánh trực tiếp.

In [ ]:
import pickle
import scipy.sparse as sp
from scipy.sparse import csr_matrix

R1_DATA_ROOT = RLMREC_ROOT / "data"
R1_NATIVE_CKPT_DIR = RLMREC_ROOT / "encoder" / "checkpoint" / "lightgcn_gene"

def r1_dataset_name(d: int) -> str:
    return f"ml20m_sub{d}_ours"

# ── (1) prepare R1 data ─────────────────────────────────────────────────────────
def _build_sparse(csv: Path, n_users: int, n_items: int) -> csr_matrix:
    df = pd.read_csv(csv)
    return csr_matrix((np.ones(len(df), dtype=np.float32), (df.userId.values, df.movieId.values)),
                      shape=(n_users, n_items))

def _align_item_emb(emb, ml20m_id_index, sub_to_ml20m, n_items):
    ml20m_to_row = {int(mid): row for row, mid in enumerate(ml20m_id_index)}
    aligned = np.zeros((n_items, emb.shape[1]), dtype=np.float32); matched = 0
    for sub_id_str, ml20m_orig in sub_to_ml20m.items():
        row = ml20m_to_row.get(int(ml20m_orig))
        if row is not None:
            aligned[int(sub_id_str)] = emb[row]; matched += 1
    return aligned, matched

def _build_user_emb(train_csv, item_emb, n_users):
    df = pd.read_csv(train_csv)
    out = np.zeros((n_users, item_emb.shape[1]), dtype=np.float32)
    for uid, g in df.groupby("userId"):
        out[uid] = item_emb[g.movieId.values].mean(axis=0)
    nrm = np.linalg.norm(out, axis=1, keepdims=True); nrm[nrm == 0] = 1.0
    return (out / nrm).astype(np.float32)

def prepare_r1_data(d: int):
    data_dir = DENSITY_DIRS[d]
    out_dir = R1_DATA_ROOT / r1_dataset_name(d); out_dir.mkdir(parents=True, exist_ok=True)
    stats = json.loads((data_dir / "stats.json").read_text())
    n_users, n_items = stats["n_users"], stats["n_items"]
    pickle.dump(_build_sparse(data_dir / "train.csv", n_users, n_items), (out_dir / "trn_mat.pkl").open("wb"))
    pickle.dump(_build_sparse(data_dir / "val.csv",   n_users, n_items), (out_dir / "val_mat.pkl").open("wb"))
    pickle.dump(_build_sparse(data_dir / "test.csv",  n_users, n_items), (out_dir / "tst_mat.pkl").open("wb"))
    profile_emb = np.load(EMBEDDING_DIR / "profile_embeddings.npy")
    ml20m_ids   = json.loads((EMBEDDING_DIR / "movie_id_index.json").read_text())
    sub_to_ml20m = json.loads((data_dir / "subsampled_to_ml20m_movieId.json").read_text())
    item_emb, matched = _align_item_emb(profile_emb, ml20m_ids, sub_to_ml20m, n_items)
    pickle.dump(item_emb, (out_dir / "itm_emb_np.pkl").open("wb"))
    pickle.dump(_build_user_emb(data_dir / "train.csv", item_emb, n_users), (out_dir / "usr_emb_np.pkl").open("wb"))
    pickle.dump([{"profile": "", "reasoning": ""} for _ in range(n_items)], (out_dir / "itm_prf.pkl").open("wb"))
    pickle.dump([{"profile": "", "reasoning": ""} for _ in range(n_users)], (out_dir / "usr_prf.pkl").open("wb"))
    logger.info(f"[R1 prep sub{d}] {out_dir.relative_to(REPO)}  aligned {matched}/{n_items} items")

# ── (2a) patch data handler (thêm nhánh predir, idempotent, không phá lõi) ──────
def patch_data_handler_for(d: int):
    name = r1_dataset_name(d)
    text = DATA_HANDLER.read_text()
    if f"'{name}'" in text or f'"{name}"' in text:
        return
    needle = "        else:\n            raise NotImplementedError"
    branch = (f"        elif configs['data']['name'] == '{name}':\n"
              f"            predir = './data/{name}/'\n")
    if needle in text:
        text = text.replace(needle, branch + needle, 1)
        DATA_HANDLER.write_text(text)
        logger.info(f"[R1] patched data_handler (+{name})")
    else:
        logger.warning(f"[R1] không tìm anchor trong data_handler để thêm {name}")

# ── (2b) modelconf YAML riêng cho (d, seed) — có ckpt_out_dir/resume ────────────
def write_r1_modelconf(d: int, seed: int) -> Path:
    import yaml
    base = yaml.safe_load(LIGHTGCN_GENE_YML.read_text())
    name = r1_dataset_name(d)
    base["model"]["name"] = "lightgcn_gene"
    base["model"]["embedding_size"] = R1_EMBEDDING_SIZE
    base["model"][name] = {
        "layer_num": R1_LAYER_NUM, "reg_weight": R1_REG_WEIGHT,
        "mask_ratio": R1_MASK_RATIO, "recon_weight": R1_RECON_WEIGHT,
        "re_temperature": R1_RE_TEMPERATURE,
    }
    base["data"]["name"] = name
    exp_name = f"r1/{ENCODER_NAME}/seed-{seed}"
    ckpt_out = (CKPT_ROOT / f"sub{d}" / exp_name)
    ckpt_out.mkdir(parents=True, exist_ok=True)
    base["train"]["seed"] = seed
    base["train"]["ckpt_out_dir"] = str(ckpt_out)   # → best_model.pt + training_state.pt
    base["train"]["resume"] = True
    yml_path = R1_MODELCONF / f"lightgcn_gene_{name}_seed{seed}.yml"
    yml_path.write_text(yaml.safe_dump(base, sort_keys=False, allow_unicode=True))
    return yml_path

# ── (3a) train R1 ───────────────────────────────────────────────────────────────
def train_R1(d: int, seed: int, log_path: Path) -> float:
    name = r1_dataset_name(d)
    yml_path = write_r1_modelconf(d, seed)
    cmd = [sys.executable, "encoder/train_encoder.py",
           "--model", "lightgcn_gene", "--dataset", name,
           "--model-conf", str(yml_path),
           "--device", DEVICE, "--seed", str(seed)]
    t0 = time.time()
    with open(log_path, "w") as lf:
        proc = subprocess.run(cmd, cwd=str(RLMREC_ROOT), stdout=lf, stderr=subprocess.STDOUT, text=True)
    wall = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"R1 sub{d} seed{seed} train FAILED (exit={proc.returncode}); xem {log_path}")
    return wall

# ── (3b) eval R1 bằng compute_metrics của repo (mô phỏng eval_ml20m_sub163_r1.py) ─
from collections import defaultdict
_BENCH_ON_PATH = str(BENCH)
if _BENCH_ON_PATH not in sys.path:
    sys.path.insert(0, _BENCH_ON_PATH)

def _r1_build_norm_adj(n_users, n_items, train_user_items, device):
    rows, cols = [], []
    for u, items in train_user_items.items():
        for it in items:
            rows.append(u); cols.append(it)
    import torch
    inter = csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(n_users, n_items))
    a = csr_matrix((n_users, n_users)); b = csr_matrix((n_items, n_items))
    bip = sp.vstack([sp.hstack([a, inter]), sp.hstack([inter.T, b])]).tocsr()
    bip = (bip != 0).astype(np.float32)
    deg = np.array(bip.sum(axis=1)).flatten()
    dis = np.power(deg, -0.5, where=deg > 0, out=np.zeros_like(deg)); dis[np.isinf(dis)] = 0.0
    norm = sp.diags(dis) @ bip @ sp.diags(dis)
    coo = norm.tocoo()
    idx = torch.LongTensor(np.vstack([coo.row, coo.col])); val = torch.FloatTensor(coo.data)
    return torch.sparse_coo_tensor(idx, val, coo.shape, device=device).coalesce()

def _r1_propagate(adj, user_emb, item_emb, layer_num):
    import torch
    embeds = torch.cat([user_emb, item_emb], dim=0); embeds_list = [embeds]
    for _ in range(layer_num):
        embeds = torch.sparse.mm(adj, embeds_list[-1]); embeds_list.append(embeds)
    final = sum(embeds_list)
    return final[: user_emb.shape[0]], final[user_emb.shape[0]:]

def eval_R1(d: int, seed: int) -> dict:
    import torch
    from data.dataset import InteractionData
    from evaluate import compute_metrics
    TOP_K = [10, 20, 50]
    name = r1_dataset_name(d)
    data = InteractionData(data_dir=DENSITY_DIRS[d])
    adj = _r1_build_norm_adj(data.n_users, data.n_items, data.train_user_items, DEVICE)
    ck = R1_NATIVE_CKPT_DIR / f"lightgcn_gene-{name}-{seed}.pth"
    sd = torch.load(ck, map_location=DEVICE, weights_only=False)
    user_emb = sd["user_embeds"].to(DEVICE); item_emb = sd["item_embeds"].to(DEVICE)
    uf, itf = _r1_propagate(adj, user_emb, item_emb, R1_LAYER_NUM)
    eval_users = sorted(data.test_user_items.keys())
    all_m = defaultdict(list); bs = 256
    with torch.no_grad():
        for s in range(0, len(eval_users), bs):
            bu = eval_users[s:s+bs]
            ur = uf[torch.LongTensor(bu).to(DEVICE)]
            scores = (ur @ itf.T).cpu().numpy()
            for i, uid in enumerate(bu):
                gt = data.test_user_items.get(uid, set())
                if not gt: continue
                tr = data.train_user_items.get(uid, set())
                for k, v in compute_metrics(scores[i], gt, tr, data.n_items, TOP_K).items():
                    all_m[k].append(v)
    m = {k: float(np.mean(v)) for k, v in all_m.items()}
    # best_epoch (nếu có training_state.pt resumable)
    best_epoch = None
    ts = CKPT_ROOT / f"sub{d}" / f"r1/{ENCODER_NAME}/seed-{seed}" / "training_state.pt"
    try:
        best_epoch = int(torch.load(ts, map_location="cpu", weights_only=False).get("best_epoch"))
    except Exception:
        pass
    out = {
        "ndcg10": m.get("NDCG@10"), "recall10": m.get("Recall@10"), "mrr": m.get("MRR"),
        "ndcg20": m.get("NDCG@20"), "recall20": m.get("Recall@20"),
        "best_epoch": best_epoch, "native_ckpt": str(ck),
    }
    res_dir = RESULTS_ROOT / f"sub{d}" / "r1"; res_dir.mkdir(parents=True, exist_ok=True)
    (res_dir / f"seed-{seed}-metrics.json").write_text(json.dumps(out, indent=2))
    return out

def run_R1_cell(d: int, seed: int) -> dict:
    log_dir = LOG_ROOT / f"sub{d}"; log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"R1-seed{seed}.log"
    prepare_r1_data(d)
    patch_data_handler_for(d)
    wall = train_R1(d, seed, log_path)
    out = eval_R1(d, seed)
    out["train_time_sec"] = round(wall, 1); out["wall_sec"] = round(wall, 1)
    out["ckpt_path"] = str(CKPT_ROOT / f"sub{d}" / f"r1/{ENCODER_NAME}/seed-{seed}" / "best_model.pt")
    return out


## 6 · Sweep (4 × 3 × 3 = 36 ô)

Idempotent: ô đã `ok` trong `run_records.jsonl` sẽ bị **bỏ qua** khi chạy lại (trừ khi `RERUN=True`).
Có thể bỏ giữa chừng rồi chạy lại — checkpoint M1/M7/R1 đều resume được.

In [ ]:
def _load_records() -> dict:
    recs = {}
    if RUN_RECORDS.exists():
        for line in RUN_RECORDS.read_text().splitlines():
            line = line.strip()
            if not line: continue
            r = json.loads(line)
            recs[(r["density"], r["method"], r["seed"])] = r
    return recs

def _append_record(rec: dict):
    with open(RUN_RECORDS, "a") as f:
        f.write(json.dumps(rec, default=str) + "\n")

records = {} if RERUN else _load_records()
if RERUN and RUN_RECORDS.exists():
    RUN_RECORDS.unlink()   # xoá sạch để ghi lại từ đầu

total = len(DENSITIES) * len(METHODS) * len(SEEDS)
done = 0
logger.info(f"=== SWEEP bắt đầu: {total} ô (device={DEVICE}) ===")
for d in DENSITIES:
    for method in METHODS:
        for seed in SEEDS:
            key = (d, method, seed)
            done += 1
            if (not RERUN) and key in records and records[key].get("status") == "ok":
                logger.info(f"[{done}/{total}] sub{d} {method} seed{seed} — đã ok, bỏ qua")
                continue
            logger.info(f"[{done}/{total}] === sub{d} {method} seed{seed} ===")
            t0 = time.time()
            try:
                metrics = run_M_cell(method, d, seed) if method in M_SPEC else run_R1_cell(d, seed)
                ach = json.loads((DENSITY_DIRS[d] / "stats.json").read_text())["achieved_int_per_item"]
                rec = {"density": d, "achieved_int_per_item": round(ach, 1),
                       "method": method, "seed": seed, "status": "ok",
                       "elapsed_sec": round(time.time() - t0, 1), **metrics}
                logger.info(f"    NDCG@10={metrics['ndcg10']:.4f}  Recall@10={metrics['recall10']:.4f}  "
                            f"MRR={metrics['mrr']:.4f}  ({rec['elapsed_sec']}s)")
            except Exception as e:
                rec = {"density": d, "method": method, "seed": seed, "status": "error",
                       "error": str(e), "elapsed_sec": round(time.time() - t0, 1)}
                logger.error(f"    LỖI: {e}")
            records[key] = rec
            _append_record(rec)
logger.info("=== SWEEP xong ===")


17:21:44 [INFO] === SWEEP bắt đầu: 36 ô (device=cpu) ===
17:21:44 [INFO] [1/36] === sub600 M1 seed42 ===


## 7 · Tổng hợp

Đọc lại `run_records.jsonl` → `per_run_results.csv` (bảng phẳng) → `summary_mean_std.csv`
(mean ± std theo (mật độ, method)) → `delta_R1_vs_M7.csv` (∆R1 vs M7 theo NDCG@10).

In [ ]:
import pandas as pd, numpy as np

recs = []
for line in RUN_RECORDS.read_text().splitlines():
    line = line.strip()
    if line: recs.append(json.loads(line))
# giữ bản ghi mới nhất cho mỗi (density, method, seed)
latest = {}
for r in recs:
    latest[(r["density"], r["method"], r["seed"])] = r
df = pd.DataFrame([r for r in latest.values() if r.get("status") == "ok"])

if df.empty:
    print("Chưa có ô nào 'ok' — hãy chạy ô Sweep trước.")
else:
    cols = ["density", "achieved_int_per_item", "method", "seed",
            "ndcg10", "recall10", "mrr", "ndcg20", "recall20",
            "best_epoch", "train_time_sec", "ckpt_path"]
    per_run = df[[c for c in cols if c in df.columns]].sort_values(["density", "method", "seed"])
    per_run.to_csv(EXP_ROOT / "per_run_results.csv", index=False)

    g = per_run.groupby(["density", "achieved_int_per_item", "method"])
    summary = g.agg(
        ndcg10_mean=("ndcg10", "mean"), ndcg10_std=("ndcg10", lambda x: x.std(ddof=1)),
        recall10_mean=("recall10", "mean"), recall10_std=("recall10", lambda x: x.std(ddof=1)),
        mrr_mean=("mrr", "mean"), mrr_std=("mrr", lambda x: x.std(ddof=1)),
        n_seeds=("seed", "count"),
    ).reset_index().sort_values(["density", "method"])
    summary.to_csv(EXP_ROOT / "summary_mean_std.csv", index=False)

    # ∆R1 vs M7 (% NDCG@10) theo mật độ
    delta_rows = []
    for (d, ach), sub in summary.groupby(["density", "achieved_int_per_item"]):
        m7 = sub[sub.method == "M7"]["ndcg10_mean"]
        r1 = sub[sub.method == "R1"]["ndcg10_mean"]
        if len(m7) and len(r1) and float(m7.iloc[0]) != 0:
            m7v, r1v = float(m7.iloc[0]), float(r1.iloc[0])
            delta_rows.append({"density": d, "achieved_int_per_item": ach,
                               "M7_ndcg10": round(m7v, 4), "R1_ndcg10": round(r1v, 4),
                               "delta_R1_vs_M7_pct": round((r1v - m7v) / m7v * 100, 2)})
    delta = pd.DataFrame(delta_rows).sort_values("achieved_int_per_item", ascending=False)
    delta.to_csv(EXP_ROOT / "delta_R1_vs_M7.csv", index=False)

    print("── per_run_results.csv ──"); print(per_run.to_string(index=False))
    print("\n── summary_mean_std.csv ──"); print(summary.to_string(index=False))
    print("\n── delta_R1_vs_M7.csv ──"); print(delta.to_string(index=False))


## 8 · Vẽ ∆R1(d)

Hai panel: (trái) **∆R1 vs M7 (% NDCG@10)** theo `log(achieved int/item)` — kỳ vọng đơn điệu (mật độ
càng thấp, R1 càng thắng); (phải) **NDCG@10** của M1/M7/R1 theo `log(d)`. Lưu `density_sweep_plot.png`.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

summary_path = EXP_ROOT / "summary_mean_std.csv"
delta_path   = EXP_ROOT / "delta_R1_vs_M7.csv"
if not summary_path.exists():
    print("Chưa có summary_mean_std.csv — chạy ô Tổng hợp trước.")
else:
    summary = pd.read_csv(summary_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # (trái) ∆R1 vs M7
    if delta_path.exists() and len(pd.read_csv(delta_path)):
        delta = pd.read_csv(delta_path).sort_values("achieved_int_per_item")
        x = np.log(delta["achieved_int_per_item"].values)
        y = delta["delta_R1_vs_M7_pct"].values
        ax1.axhline(0, color="grey", lw=0.8, ls="--")
        ax1.plot(x, y, "o-", color="#c0392b", lw=2, ms=8)
        for xi, yi, di in zip(x, y, delta["achieved_int_per_item"].values):
            ax1.annotate(f"{yi:+.1f}%\n(d≈{di:.0f})", (xi, yi),
                         textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
    ax1.set_xlabel("log(achieved int/item)"); ax1.set_ylabel("∆R1 vs M7 (% NDCG@10)")
    ax1.set_title("∆R1(d): mật độ càng thấp → R1 càng thắng?"); ax1.grid(alpha=0.3)

    # (phải) NDCG@10 theo log(d) cho M1/M7/R1
    colors = {"M1": "#2980b9", "M7": "#27ae60", "R1": "#c0392b"}
    for method, sub in summary.groupby("method"):
        sub = sub.sort_values("achieved_int_per_item")
        x = np.log(sub["achieved_int_per_item"].values)
        ax2.errorbar(x, sub["ndcg10_mean"].values, yerr=sub["ndcg10_std"].values,
                     marker="o", capsize=3, lw=2, label=method,
                     color=colors.get(method))
    ax2.set_xlabel("log(achieved int/item)"); ax2.set_ylabel("NDCG@10 (mean ± std)")
    ax2.set_title("NDCG@10 theo mật độ"); ax2.legend(); ax2.grid(alpha=0.3)

    fig.tight_layout()
    out_png = EXP_ROOT / "density_sweep_plot.png"
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    print(f"Đã lưu {out_png}")
    try:
        from IPython.display import Image, display
        display(Image(filename=str(out_png)))
    except Exception:
        pass
